In [1]:
import re
import ast
import json
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/basmala.ayman/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Helper Functions

In [2]:
stop_words = set(stopwords.words("english"))

def tokenize_words(text: str):
    text = "" if pd.isna(text) else str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

In [3]:
def to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x = str(x).strip()
    if x == "" or x == "[]":
        return []
    # if it looks like a python list
    if x.startswith("[") and x.endswith("]"):
        try:
            return ast.literal_eval(x)
        except:
            return []
    # otherwise treat as single token
    return [x]

In [4]:
def clean_label(tok: str):
    tok = "" if tok is None else str(tok).lower().strip()
    tok = re.sub(r"\s+", " ", tok)
    # keep + # . - for c++, c#, node.js, etc.
    tok = re.sub(r"[^a-z0-9\+\#\.\- ]", " ", tok)
    tok = re.sub(r"\s+", " ", tok).strip()
    return tok

In [5]:
def uniq_clean_list(lst):
    if not isinstance(lst, list):
        return []
    out, seen = [], set()
    for x in lst:
        x = clean_label(x)
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out

## Load Dataset

In [6]:
teams   = pd.read_csv("datasets/without_cold/teams_without_cold.csv")
members = pd.read_csv("datasets/without_cold/mapped_members_without_cold.csv")

In [7]:
teams.columns

Index(['project_id', 'hackathon_id', 'members_count', 'is_winner',
       'winners_description', 'project_tags', 'project_description',
       'short_description'],
      dtype='str')

In [8]:
members['hackathons_interests'] = members['hackathons_interests'].apply(to_list).apply(uniq_clean_list)
members['hard_skills_clean'] = members['hard_skills_clean'].apply(to_list).apply(uniq_clean_list)
members['mapped_soft_skills'] = members['mapped_soft_skills'].apply(to_list).apply(uniq_clean_list)

teams["project_tags"] = teams["project_tags"].apply(to_list).apply(uniq_clean_list)

In [9]:
members["member_id"] = members["member_id"].astype(int)
teams["project_id"] = teams["project_id"].astype(int)

In [10]:
teams["short_description"] = teams["short_description"].fillna("").astype(str)

members['bio'] = members['bio'].fillna("").astype(str)
# convert it to a list, and each element is a string
members["Descriptions"] = members["Descriptions"].apply(to_list).apply(
    lambda lst: [str(x) for x in lst if str(x).strip() != ""]
)

In [11]:
print("Members:", members.shape, "Projects:", teams.shape)

Members: (38462, 14) Projects: (18569, 8)


## Build vocabularies

In [12]:
PAD = "<PAD>"
UNK = "<UNK>"

word_counter = Counter()

# words from task short_description
for txt in teams['short_description'].tolist():
    word_counter.update(tokenize_words(txt))

# words from member bio + Descriptions
for _, row in members.iterrows():
    bio = row.get("bio", "")
    desc_list = row.get("Descriptions", [])
    desc_text = " ".join(desc_list) if isinstance(desc_list, list) else ""
    full_text = f"{bio} {desc_text}"
    word_counter.update(tokenize_words(full_text))

MAX_WORDS = 50000
most_common_words = [w for w, _ in word_counter.most_common(MAX_WORDS)]

# build mapping
word2id = {PAD: 0, UNK: 1}
for w in most_common_words:
    if w not in word2id:
        word2id[w] = len(word2id)

print("Word vocab size:", len(word2id))
print("Sample words:", list(word2id.items())[:10])

Word vocab size: 29261
Sample words: [('<PAD>', 0), ('<UNK>', 1), ('worked', 2), ('end', 3), ('using', 4), ('backend', 5), ('front', 6), ('also', 7), ('project', 8), ('time', 9)]


## Build label2id

In [13]:
label_counter = Counter()

# project tags (built-with)
for tags in teams["project_tags"].tolist():
    label_counter.update(tags)

# member labels
for col in ["hard_skills_clean", "mapped_soft_skills", "hackathons_interests"]:
    for lst in members[col].tolist():
        label_counter.update(lst)


MIN_LABEL_FREQ = 2

label2id = {PAD: 0, UNK: 1}
for lbl, cnt in label_counter.items():
    if cnt >= MIN_LABEL_FREQ:
        label2id[lbl] = len(label2id)

print("Label vocab size:", len(label2id))
print("Sample labels:", list(label2id.items())[:10])

Label vocab size: 4791
Sample labels: [('<PAD>', 0), ('<UNK>', 1), ('css', 2), ('firebase', 3), ('flask', 4), ('html', 5), ('javascript', 6), ('python', 7), ('react', 8), ('cadence', 9)]


## Save vocabularies

In [14]:
with open("embeddings/word2id.json", "w") as f:
    json.dump(word2id, f)

with open("embeddings/label2id.json", "w") as f:
    json.dump(label2id, f)

print("Saved vocab files")

Saved vocab files


## Padding

In [15]:
L_TITLE     = 40    # words from short_description
L_TAGS      = 20    # built-with tags
L_MEMBER_TEXT  = 120   # words from bio + descriptions (increase if you want)
L_HARD      = 40
L_SOFT      = 20
L_INT       = 15

In [16]:
def pad_ids(ids, max_len, pad_id=0):
    ids = ids[:max_len]
    mask = [1] * len(ids)
    while len(ids) < max_len:
        ids.append(pad_id)
        mask.append(0)
    return ids, mask

## Build Project tensors (title + tags)

In [17]:
project_ids = teams["project_id"].tolist()

title_ids_mat, title_mask_mat = [], []
tag_ids_mat, tag_mask_mat     = [], []

for _, row in teams.iterrows():
    # title words
    words = tokenize_words(row["short_description"])
    w_ids = [word2id.get(w, word2id[UNK]) for w in words]
    w_ids, w_mask = pad_ids(w_ids, L_TITLE, pad_id=word2id[PAD])

    # built-with tags (labels)
    tags = row["project_tags"]
    t_ids = [label2id.get(t, label2id[UNK]) for t in tags]
    t_ids, t_mask = pad_ids(t_ids, L_TAGS, pad_id=label2id[PAD])

    title_ids_mat.append(w_ids);  title_mask_mat.append(w_mask)
    tag_ids_mat.append(t_ids);    tag_mask_mat.append(t_mask)

title_ids  = torch.tensor(title_ids_mat, dtype=torch.long)
title_mask = torch.tensor(title_mask_mat, dtype=torch.bool)

tag_ids  = torch.tensor(tag_ids_mat, dtype=torch.long)
tag_mask = torch.tensor(tag_mask_mat, dtype=torch.bool)

print("title_ids:", title_ids.shape, "title_mask:", title_mask.shape)
print("tag_ids:", tag_ids.shape, "tag_mask:", tag_mask.shape)

title_ids: torch.Size([18569, 40]) title_mask: torch.Size([18569, 40])
tag_ids: torch.Size([18569, 20]) tag_mask: torch.Size([18569, 20])


## Build Member tensors

In [18]:
member_ids = members["member_id"].tolist()

member_text_ids_mat, member_text_mask_mat = [], []
hard_ids_mat, hard_mask_mat         = [], []
soft_ids_mat, soft_mask_mat         = [], []
int_ids_mat, int_mask_mat           = [], []

for _, row in members.iterrows():
    # member text = bio + descriptions (word vocab)
    bio = row.get("bio", "")
    desc_list = row.get("Descriptions", [])
    desc_text = " ".join(desc_list) if isinstance(desc_list, list) else ""
    full_text = f"{bio} {desc_text}".strip()

    words = tokenize_words(full_text)
    dt_ids = [word2id.get(w, word2id[UNK]) for w in words]
    dt_ids, dt_mask = pad_ids(dt_ids, L_MEMBER_TEXT, pad_id=word2id[PAD])

    # hard skills
    hard = row.get("hard_skills_clean", [])
    h_ids = [label2id.get(t, label2id[UNK]) for t in hard]
    h_ids, h_mask = pad_ids(h_ids, L_HARD, pad_id=label2id[PAD])

    # soft skills
    soft = row.get("mapped_soft_skills", [])
    s_ids = [label2id.get(t, label2id[UNK]) for t in soft]
    s_ids, s_mask = pad_ids(s_ids, L_SOFT, pad_id=label2id[PAD])

    # hackathons interests
    ints = row.get("hackathons_interests", [])
    i_ids = [label2id.get(t, label2id[UNK]) for t in ints]
    i_ids, i_mask = pad_ids(i_ids, L_INT, pad_id=label2id[PAD])

    member_text_ids_mat.append(dt_ids); member_text_mask_mat.append(dt_mask)
    hard_ids_mat.append(h_ids);      hard_mask_mat.append(h_mask)
    soft_ids_mat.append(s_ids);      soft_mask_mat.append(s_mask)
    int_ids_mat.append(i_ids);       int_mask_mat.append(i_mask)

member_text_ids  = torch.tensor(member_text_ids_mat, dtype=torch.long)
member_text_mask = torch.tensor(member_text_mask_mat, dtype=torch.bool)

hard_ids  = torch.tensor(hard_ids_mat, dtype=torch.long)
hard_mask = torch.tensor(hard_mask_mat, dtype=torch.bool)

soft_ids  = torch.tensor(soft_ids_mat, dtype=torch.long)
soft_mask = torch.tensor(soft_mask_mat, dtype=torch.bool)

int_ids  = torch.tensor(int_ids_mat, dtype=torch.long)
int_mask = torch.tensor(int_mask_mat, dtype=torch.bool)

print("member_text_ids:", member_text_ids.shape, "member_text_mask:", member_text_mask.shape)
print("hard_ids:", hard_ids.shape, "soft_ids:", soft_ids.shape, "int_ids:", int_ids.shape)

member_text_ids: torch.Size([38462, 120]) member_text_mask: torch.Size([38462, 120])
hard_ids: torch.Size([38462, 40]) soft_ids: torch.Size([38462, 20]) int_ids: torch.Size([38462, 15])


In [19]:
torch.save(
    {
        "project_ids": project_ids,
        "title_ids": title_ids,
        "title_mask": title_mask,
        "tag_ids": tag_ids,
        "tag_mask": tag_mask,
        "member_ids": member_ids,
        "member_text_ids": member_text_ids,
        "member_text_mask": member_text_mask,
        "hard_ids": hard_ids,
        "hard_mask": hard_mask,
        "soft_ids": soft_ids,
        "soft_mask": soft_mask,
        "int_ids": int_ids,
        "int_mask": int_mask,
    },
    "embeddings/torch_tensor_words.pt"
)